# BACI 2023: empirical Slack, tail diagnostics, and lognormal calibration

Reproducibility notebook for the empirical component of **Distributional Sensitivity of Slack-Based Fragility Measures in Weighted Bipartite Networks**.

**Workflow:** load BACI 2023 trade data, construct exporter–HS6 weighted bipartite networks for six macro-sectors, calculate Slack and pivotality, fit tail diagnostics, generate paper-facing tables and CCDFs, then calibrate the simulation's lognormal dispersion to observed BACI target concentration **under the Monte Carlo Bernoulli topology**.

**Data:** BACI source CSVs and code tables are not bundled with this notebook. Run in Google Colab, mount Drive, and set `DATA_DIR` in the configuration cell to the directory containing your BACI files. Execute cells from top to bottom. Output files are written under `slack_empirical_outputs/`.

**Calibration note:** The final topology-adjusted lognormal calibration is the one used in `simulation.py` (sigma = 1.45 in the revised paper). Exploratory intermediate calibration procedures have been omitted to avoid confusion; the empirical target-share concentration statistics are preserved.


In [ ]:
!pip -q install powerlaw

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import csv
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# ============================================================
# CHANGE THESE SETTINGS IF NEEDED
# ============================================================

DATA_DIR = Path("/content/drive/MyDrive/Simulation Framework Slack (log analysis)")

YEARS = [2023]  # For COVID: [2019, 2020, 2021]; for war/energy shock: [2021, 2022, 2023]
SHOCK_NAME = "single_year_2023"

# YEARS = [2019, 2020, 2021]  # For COVID: ; for war/energy shock: [2021, 2022, 2023]
# SHOCK_NAME = "covid"

THETA_VALUES = [0.50, 0.70, 0.90]

# "product" = exporter -> HS6 product
# "importer_product" = exporter -> importer-HS6 product market
TARGET_MODE = "product"

BACI_VERSION = "V202601"
HS_VERSION = "HS17"

country_file = DATA_DIR / "country_codes_V202601.csv"
product_file = DATA_DIR / "product_codes_HS17_V202601.csv"

OUTPUT_DIR = DATA_DIR / "slack_empirical_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def baci_path(year):
    return DATA_DIR / f"BACI_{HS_VERSION}_Y{year}_{BACI_VERSION}.csv"

print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
for y in YEARS:
    print(y, baci_path(y), "exists:", baci_path(y).exists())
print("country_file exists:", country_file.exists())
print("product_file exists:", product_file.exists())

In [ ]:
# ============================================================
# MACRO-SECTOR CLASSIFICATION FROM HS CHAPTERS
# ============================================================

SELECTED_SECTORS = [
    "Energy",
    "Agri-food",
    "Chemicals",
    "Metals/minerals",
    "Machinery/electronics",
    "Transport"
]

def assign_macro_sector(chapter):
    if 1 <= chapter <= 24:
        return "Agri-food"
    elif chapter == 27:
        return "Energy"
    elif 28 <= chapter <= 38:
        return "Chemicals"
    elif chapter in [25, 26] or 72 <= chapter <= 83:
        return "Metals/minerals"
    elif 84 <= chapter <= 85:
        return "Machinery/electronics"
    elif 86 <= chapter <= 89:
        return "Transport"
    else:
        return None

In [ ]:
# ============================================================
# OPTIONAL METADATA
# ============================================================

def pick_column(cols, possible_names):
    cols_lower = {str(c).lower(): c for c in cols}
    for name in possible_names:
        if name.lower() in cols_lower:
            return cols_lower[name.lower()]
    for c in cols:
        cl = str(c).lower()
        if any(name.lower() in cl for name in possible_names):
            return c
    return None

def load_country_map(path):
    if not Path(path).exists():
        print("Country file not found. Country names will not be used.")
        return {}
    countries = pd.read_csv(path)
    print("Country columns:", countries.columns.tolist())
    display(countries.head())
    code_col = pick_column(countries.columns, ["country_code", "code", "i"])
    name_col = pick_column(countries.columns, ["country_name", "name", "country"])
    if code_col is None or name_col is None:
        print("Could not automatically identify country code/name columns.")
        return {}
    countries[code_col] = pd.to_numeric(countries[code_col], errors="coerce")
    countries = countries.dropna(subset=[code_col])
    countries[code_col] = countries[code_col].astype(int)
    return dict(zip(countries[code_col], countries[name_col].astype(str)))

def load_product_map(path):
    if not Path(path).exists():
        print("Product file not found. Product descriptions will not be used.")
        return {}
    products = pd.read_csv(path, dtype=str)
    print("Product columns:", products.columns.tolist())
    display(products.head())
    code_col = pick_column(products.columns, ["code", "product_code", "k"])
    desc_col = pick_column(products.columns, ["description", "product_description", "name"])
    if code_col is None or desc_col is None:
        print("Could not automatically identify product code/description columns.")
        return {}
    products["hs6_clean"] = products[code_col].astype(str).str.replace(r"\.0$", "", regex=True).str.zfill(6)
    return dict(zip(products["hs6_clean"], products[desc_col].astype(str)))

country_map = load_country_map(country_file)
product_map = load_product_map(product_file)

print("Countries loaded:", len(country_map))
print("Products loaded:", len(product_map))

In [ ]:
# ============================================================
# ROBUST BACI LOADER
# ============================================================

def detect_delimiter(file_path):
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        sample = f.read(4096)
    try:
        return csv.Sniffer().sniff(sample, delimiters=",;\t").delimiter
    except Exception:
        return ","

def load_baci_edges_by_sector(year, target_mode="product", chunksize=1_000_000):
    file_path = baci_path(year)
    if not file_path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

    sep = detect_delimiter(file_path)
    print(f"\nLoading {file_path}")
    print("Detected separator:", repr(sep))

    header = pd.read_csv(file_path, sep=sep, nrows=0)
    header.columns = [str(c).strip() for c in header.columns]
    print("Detected columns:", header.columns.tolist())

    required = {"i", "j", "k", "v"}
    missing = required.difference(set(header.columns))
    if missing:
        raise ValueError(f"Missing required columns {missing}. Detected columns: {header.columns.tolist()}")

    usecols = ["i", "j", "k", "v"]
    agg_parts = []

    reader = pd.read_csv(
        file_path,
        sep=sep,
        usecols=usecols,
        dtype={"k": str},
        chunksize=chunksize
    )

    for chunk_id, chunk in enumerate(reader):
        chunk.columns = [str(c).strip() for c in chunk.columns]

        chunk["v"] = pd.to_numeric(chunk["v"], errors="coerce")
        chunk["i"] = pd.to_numeric(chunk["i"], errors="coerce")
        chunk["j"] = pd.to_numeric(chunk["j"], errors="coerce")
        chunk = chunk.dropna(subset=["i", "j", "k", "v"])
        chunk = chunk[chunk["v"] > 0].copy()

        if chunk.empty:
            continue

        chunk["i"] = chunk["i"].astype(int)
        chunk["j"] = chunk["j"].astype(int)

        # BACI may store HS codes like 10121 instead of 010121, so zfill(6) is essential.
        chunk["hs6"] = (
            chunk["k"].astype(str)
            .str.replace(r"\.0$", "", regex=True)
            .str.strip()
            .str.zfill(6)
        )

        chunk["chapter"] = pd.to_numeric(chunk["hs6"].str[:2], errors="coerce")
        chunk = chunk.dropna(subset=["chapter"])
        chunk["chapter"] = chunk["chapter"].astype(int)

        chunk["macro_sector"] = chunk["chapter"].apply(assign_macro_sector)
        chunk = chunk[chunk["macro_sector"].isin(SELECTED_SECTORS)].copy()

        if chunk.empty:
            continue

        if target_mode == "product":
            g = (
                chunk.groupby(["i", "macro_sector", "hs6"], as_index=False)["v"]
                .sum()
                .rename(columns={"i": "source", "v": "weight"})
            )
            g["target"] = g["macro_sector"] + ":" + g["hs6"]
            g = g[["source", "target", "macro_sector", "hs6", "weight"]]

        elif target_mode == "importer_product":
            chunk["target"] = chunk["j"].astype(str) + ":" + chunk["macro_sector"] + ":" + chunk["hs6"]
            g = (
                chunk.groupby(["i", "j", "target", "macro_sector", "hs6"], as_index=False)["v"]
                .sum()
                .rename(columns={"i": "source", "j": "importer", "v": "weight"})
            )
            g = g[["source", "importer", "target", "macro_sector", "hs6", "weight"]]
        else:
            raise ValueError("target_mode must be 'product' or 'importer_product'.")

        agg_parts.append(g)

        if chunk_id % 5 == 0:
            print(f"  processed chunk {chunk_id}; accumulated parts: {len(agg_parts)}")

    if not agg_parts:
        raise ValueError(f"No selected-sector data found for year {year}.")

    edges = pd.concat(agg_parts, ignore_index=True)
    group_cols = [c for c in edges.columns if c != "weight"]

    edges = (
        edges.groupby(group_cols, as_index=False)["weight"]
        .sum()
        .sort_values("weight", ascending=False)
        .reset_index(drop=True)
    )

    print(f"Year {year} aggregated edges:", edges.shape)
    display(edges.head())

    return edges

In [ ]:
# ============================================================
# SLACK FUNCTIONS
# ============================================================

def gini(x):
    x = np.asarray(pd.Series(x).dropna(), dtype=float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return np.nan
    if np.all(x == 0):
        return 0.0
    x = np.sort(x)
    n = len(x)
    return (2 * np.sum(np.arange(1, n + 1) * x) / (n * np.sum(x))) - (n + 1) / n

def compute_slack_for_sector(edges_sector, theta, year, macro_sector):
    edges = edges_sector.copy()
    edges = edges[edges["weight"] > 0].copy()

    target_totals = edges.groupby("target", as_index=False)["weight"].sum().rename(columns={"weight": "T_j"})
    edges = edges.merge(target_totals, on="target", how="left")

    edges["theta"] = theta
    edges["Q_j"] = theta * edges["T_j"]
    edges["alpha_ij"] = edges["weight"] / edges["T_j"]

    # Slack_ij < 1 means edge-level pivotality.
    edges["Slack_ij"] = (edges["T_j"] - edges["weight"]) / edges["Q_j"]
    edges["is_pivotal_edge"] = edges["Slack_ij"] < 1
    edges["criticality_depth"] = np.maximum(0, 1 - edges["Slack_ij"])

    source_totals = edges.groupby("source", as_index=False)["weight"].sum().rename(columns={"weight": "s_i"})
    edges = edges.merge(source_totals, on="source", how="left")

    edges["beta_ij"] = edges["weight"] / edges["s_i"]
    edges["weighted_slack_component"] = edges["beta_ij"] * edges["Slack_ij"]
    edges["weighted_dominance_component"] = edges["beta_ij"] * edges["alpha_ij"]

    sources = (
        edges.groupby("source")
        .agg(
            Slack_mean=("Slack_ij", "mean"),
            Slack_weighted=("weighted_slack_component", "sum"),
            dominance_score=("weighted_dominance_component", "sum"),
            TargetCount=("target", "nunique"),
            total_weight=("weight", "sum"),
            n_pivotal_edges=("is_pivotal_edge", "sum"),
            max_alpha=("alpha_ij", "max"),
            mean_alpha=("alpha_ij", "mean"),
        )
        .reset_index()
    )

    sources["is_pivotal_source"] = sources["Slack_weighted"] < 1
    sources["year"] = year
    sources["theta"] = theta
    sources["macro_sector"] = macro_sector
    sources["source_name"] = sources["source"].apply(lambda x: country_map.get(int(x), str(x)))

    edges["year"] = year
    edges["macro_sector"] = macro_sector

    summary = {
        "year": year,
        "theta": theta,
        "macro_sector": macro_sector,
        "n_sources": sources["source"].nunique(),
        "n_targets": edges["target"].nunique(),
        "n_edges": len(edges),
        "total_weight": edges["weight"].sum(),
        "frac_edges_slack_lt_one": edges["is_pivotal_edge"].mean(),
        "frac_sources_slack_weighted_lt_one": sources["is_pivotal_source"].mean(),
        "min_slack_weighted": sources["Slack_weighted"].min(),
        "median_slack_weighted": sources["Slack_weighted"].median(),
        "mean_slack_weighted": sources["Slack_weighted"].mean(),
        "gini_slack_weighted": gini(sources["Slack_weighted"]),
        "min_alpha_ij": edges["alpha_ij"].min(),
        "max_alpha_ij": edges["alpha_ij"].max(),
        "mean_alpha_ij": edges["alpha_ij"].mean(),
        "gini_alpha_ij": gini(edges["alpha_ij"]),
    }

    return edges, sources, summary

In [ ]:
# ============================================================
# RUN EMPIRICAL SLACK ANALYSIS
# ============================================================

SAVE_FULL_EDGE_SLACK = False

all_sources = []
all_summaries = []
all_top_edges = []
aggregated_edges_by_year = {}

for year in YEARS:
    edges_year = load_baci_edges_by_sector(year, target_mode=TARGET_MODE)
    aggregated_edges_by_year[year] = edges_year

    out_edges_file = OUTPUT_DIR / f"aggregated_edges_{TARGET_MODE}_{year}_{SHOCK_NAME}.csv.gz"
    edges_year.to_csv(out_edges_file, index=False, compression="gzip")
    print("Saved:", out_edges_file)

    for sector in SELECTED_SECTORS:
        edges_sector = edges_year[edges_year["macro_sector"] == sector].copy()
        if edges_sector.empty:
            print("Empty sector:", sector, year)
            continue

        for theta in THETA_VALUES:
            edges_slack, sources_slack, summary = compute_slack_for_sector(
                edges_sector=edges_sector,
                theta=theta,
                year=year,
                macro_sector=sector
            )

            all_sources.append(sources_slack)
            all_summaries.append(summary)

            top_edges = edges_slack.sort_values("Slack_ij", ascending=True).head(100).copy()
            top_edges["source_name"] = top_edges["source"].apply(lambda x: country_map.get(int(x), str(x)))
            top_edges["product_description"] = top_edges["hs6"].apply(lambda x: product_map.get(str(x).zfill(6), ""))
            all_top_edges.append(top_edges)

            if SAVE_FULL_EDGE_SLACK:
                out_full_edges = OUTPUT_DIR / f"edge_slack_{TARGET_MODE}_{year}_{sector}_theta{theta}_{SHOCK_NAME}.csv.gz"
                edges_slack.to_csv(out_full_edges, index=False, compression="gzip")

sources_df = pd.concat(all_sources, ignore_index=True)
summary_df = pd.DataFrame(all_summaries)
top_edges_df = pd.concat(all_top_edges, ignore_index=True)

sources_file = OUTPUT_DIR / f"sources_slack_{TARGET_MODE}_{SHOCK_NAME}.csv.gz"
summary_file = OUTPUT_DIR / f"summary_slack_{TARGET_MODE}_{SHOCK_NAME}.csv"
top_edges_file = OUTPUT_DIR / f"top_fragile_edges_{TARGET_MODE}_{SHOCK_NAME}.csv.gz"

sources_df.to_csv(sources_file, index=False, compression="gzip")
summary_df.to_csv(summary_file, index=False)
top_edges_df.to_csv(top_edges_file, index=False, compression="gzip")

print("\nSaved files:")
print(sources_file)
print(summary_file)
print(top_edges_file)

display(summary_df.sort_values(["theta", "macro_sector"]).head(30))

In [ ]:
# ============================================================
# POWER-LAW / HEAVY-TAIL DIAGNOSTICS
# + FULL-DISTRIBUTION LOGNORMAL CALIBRATION
# ============================================================

import powerlaw


# ------------------------------------------------------------
# 1. Add target-normalized share alpha_ij
# ------------------------------------------------------------

def add_alpha_ij(edges):
    x = edges.copy()

    target_totals = (
        x.groupby("target", as_index=False)["weight"]
        .sum()
        .rename(columns={"weight": "T_j"})
    )

    x = x.merge(target_totals, on="target", how="left")

    x["alpha_ij"] = x["weight"] / x["T_j"]

    return x


# ------------------------------------------------------------
# 2. Power-law / tail fit
#
# xmin is fixed at the q-th empirical percentile.
# Power law and lognormal are therefore compared on the
# SAME upper-tail observations x >= xmin.
# ------------------------------------------------------------

def fit_powerlaw_summary_with_LR(
    x,
    year,
    macro_sector,
    variable_name,
    q=0.90
):
    x = pd.Series(x).dropna()
    x = x[np.isfinite(x)]
    x = x[x > 0].astype(float).values

    if len(x) < 100:
        return {
            "year": year,
            "macro_sector": macro_sector,
            "variable": variable_name,
            "n": len(x),
        }

    # Fixed cutoff: 90th percentile by default
    xmin_fixed = np.percentile(x, q * 100)

    fit = powerlaw.Fit(
        x,
        xmin=xmin_fixed,
        verbose=False
    )

    row = {
        "year": year,
        "macro_sector": macro_sector,
        "variable": variable_name,
        "n": len(x),

        # Power-law fit
        "alpha_hat": fit.power_law.alpha,
        "xmin": fit.power_law.xmin,
        "KS_distance": fit.power_law.D,

        # Lognormal fitted on SAME tail x >= xmin
        "lognormal_mu_tail": fit.lognormal.mu,
        "lognormal_sigma_tail": fit.lognormal.sigma,
    }

    comparisons = {
        "lognormal": "lognormal",
        "exponential": "exponential",
        "truncated": "truncated_power_law",
    }

    for label, dist in comparisons.items():

        try:
            R, p = fit.distribution_compare(
                "power_law",
                dist
            )

            row[f"LR_powerlaw_vs_{label}"] = R
            row[f"p_powerlaw_vs_{label}"] = p

        except Exception:

            row[f"LR_powerlaw_vs_{label}"] = np.nan
            row[f"p_powerlaw_vs_{label}"] = np.nan

    return row


# ------------------------------------------------------------
# 3. Full-distribution lognormal MLE
#
# This uses ALL positive observations, not only x >= xmin.
#
# For X ~ Lognormal(mu, sigma):
#
#   mu_hat    = mean(log X)
#   sigma_hat = sd(log X), MLE version (ddof = 0)
#
# These parameters are candidates for calibrating the new
# lognormal regime in the Monte Carlo simulation.
# ------------------------------------------------------------

def fit_full_lognormal(
    x,
    year,
    macro_sector,
    variable_name
):
    x = pd.Series(x).dropna()
    x = x[np.isfinite(x)]
    x = x[x > 0].astype(float).values

    if len(x) < 2:
        return {
            "year": year,
            "macro_sector": macro_sector,
            "variable": variable_name,
            "n": len(x),
            "lognormal_mu_full": np.nan,
            "lognormal_sigma_full": np.nan,
        }

    log_x = np.log(x)

    mu_full = np.mean(log_x)
    sigma_full = np.std(log_x, ddof=0)

    # Implied arithmetic mean and SD of the fitted lognormal
    implied_mean = np.exp(
        mu_full + 0.5 * sigma_full**2
    )

    implied_sd = np.sqrt(
        (np.exp(sigma_full**2) - 1) *
        np.exp(2 * mu_full + sigma_full**2)
    )

    return {
        "year": year,
        "macro_sector": macro_sector,
        "variable": variable_name,
        "n": len(x),

        "lognormal_mu_full": mu_full,
        "lognormal_sigma_full": sigma_full,

        "lognormal_implied_mean": implied_mean,
        "lognormal_implied_sd": implied_sd,
    }


# ============================================================
# 4. Run diagnostics sector by sector
# ============================================================

tail_rows = []
lognormal_full_rows = []


for year, edges_year in aggregated_edges_by_year.items():

    for sector in SELECTED_SECTORS:

        edges_sector = edges_year[
            edges_year["macro_sector"] == sector
        ].copy()

        if edges_sector.empty:
            continue


        # --------------------------------------------------------
        # A. RAW EXPORT WEIGHTS
        # --------------------------------------------------------

        # Upper-tail power-law / lognormal diagnostics
        tail_rows.append(
            fit_powerlaw_summary_with_LR(
                edges_sector["weight"],
                year,
                sector,
                "weight"
            )
        )

        # Full-distribution lognormal calibration
        lognormal_full_rows.append(
            fit_full_lognormal(
                edges_sector["weight"],
                year,
                sector,
                "weight"
            )
        )


        # --------------------------------------------------------
        # B. TARGET-NORMALIZED SHARES alpha_ij
        # --------------------------------------------------------

        edges_alpha = add_alpha_ij(edges_sector)

        # Upper-tail diagnostics
        tail_rows.append(
            fit_powerlaw_summary_with_LR(
                edges_alpha["alpha_ij"],
                year,
                sector,
                "alpha_ij"
            )
        )

        # Full-distribution lognormal fit
        # Useful diagnostically, although the simulation weights
        # themselves will most likely be calibrated using "weight".
        lognormal_full_rows.append(
            fit_full_lognormal(
                edges_alpha["alpha_ij"],
                year,
                sector,
                "alpha_ij"
            )
        )


# ============================================================
# 5. Convert results to DataFrames
# ============================================================

tail_fit_df = pd.DataFrame(tail_rows)

lognormal_full_df = pd.DataFrame(
    lognormal_full_rows
)


# ============================================================
# 6. Save results
# ============================================================

tail_fit_file = (
    OUTPUT_DIR /
    f"tail_fit_results_{TARGET_MODE}_{SHOCK_NAME}.csv"
)

tail_fit_df.to_csv(
    tail_fit_file,
    index=False
)


lognormal_full_file = (
    OUTPUT_DIR /
    f"lognormal_full_fit_{TARGET_MODE}_{SHOCK_NAME}.csv"
)

lognormal_full_df.to_csv(
    lognormal_full_file,
    index=False
)


print("Saved:")
print(tail_fit_file)
print(lognormal_full_file)


# ============================================================
# 7. Display complete results
# ============================================================

print("\n--- TAIL FIT RESULTS ---")
display(tail_fit_df)


print("\n--- FULL LOGNORMAL FIT RESULTS ---")
display(lognormal_full_df)


# ============================================================
# 8. Display only RAW WEIGHT lognormal results
#    --> THESE ARE THE IMPORTANT ONES FOR THE SIMULATION
# ============================================================

lognormal_weight_calibration = (
    lognormal_full_df[
        lognormal_full_df["variable"] == "weight"
    ]
    .sort_values("macro_sector")
    .reset_index(drop=True)
)

print(
    "\n--- FULL-DISTRIBUTION LOGNORMAL PARAMETERS "
    "FOR RAW BACI WEIGHTS ---"
)

display(
    lognormal_weight_calibration[
        [
            "year",
            "macro_sector",
            "n",
            "lognormal_mu_full",
            "lognormal_sigma_full",
            "lognormal_implied_mean",
            "lognormal_implied_sd",
        ]
    ]
)


# ============================================================
# 9. Quick summary across sectors
# ============================================================

print("\n--- ACROSS-SECTOR SUMMARY: RAW WEIGHTS ---")

print(
    lognormal_weight_calibration[
        [
            "lognormal_mu_full",
            "lognormal_sigma_full"
        ]
    ].describe()
)

print(
    "\nMedian sigma across sectors:",
    lognormal_weight_calibration[
        "lognormal_sigma_full"
    ].median()
)

print(
    "Mean sigma across sectors:",
    lognormal_weight_calibration[
        "lognormal_sigma_full"
    ].mean()
)

In [ ]:
# ============================================================
# CLEAN SUMMARY TABLE FOR PAPER
# ============================================================

theta_paper = 0.70

paper_summary = summary_df[summary_df["theta"] == theta_paper].copy()

paper_summary = paper_summary[
    [
        "year", "macro_sector", "n_sources", "n_targets", "n_edges", "total_weight",
        "frac_edges_slack_lt_one", "frac_sources_slack_weighted_lt_one",
        "min_slack_weighted", "median_slack_weighted", "gini_slack_weighted",
        "max_alpha_ij", "gini_alpha_ij",
    ]
].sort_values(["macro_sector", "year"])

tail_short = tail_fit_df[tail_fit_df["variable"] == "weight"][
    [
        "year", "macro_sector", "alpha_hat", "xmin", "KS_distance",
        "LR_powerlaw_vs_lognormal", "p_powerlaw_vs_lognormal",
        "LR_powerlaw_vs_exponential", "p_powerlaw_vs_exponential",
    ]
]

paper_summary = paper_summary.merge(tail_short, on=["year", "macro_sector"], how="left")

paper_file = OUTPUT_DIR / f"paper_summary_theta{theta_paper}_{TARGET_MODE}_{SHOCK_NAME}.csv"
paper_summary.to_csv(paper_file, index=False)

print("Saved:", paper_file)
display(paper_summary)

In [ ]:
# ============================================================
# VISUAL DIAGNOSTIC: LOG-LOG CCDF BY SECTOR
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def ccdf_values(x):
    x = pd.Series(x).dropna()
    x = x[np.isfinite(x)]
    x = x[x > 0].astype(float)

    if len(x) == 0:
        return np.array([]), np.array([])

    x_sorted = np.sort(x.values)
    ccdf = 1.0 - np.arange(1, len(x_sorted) + 1) / len(x_sorted)

    return x_sorted, ccdf


def plot_ccdf_by_sector(year, variable="weight"):
    edges_year = aggregated_edges_by_year[year]

    plt.figure(figsize=(8, 5))

    for sector in SELECTED_SECTORS:
        edges_sector = edges_year[edges_year["macro_sector"] == sector].copy()

        if edges_sector.empty:
            continue

        if variable == "weight":
            x = edges_sector["weight"]
            xlabel = "Export value"
        elif variable == "alpha_ij":
            edges_sector = add_alpha_ij(edges_sector)
            x = edges_sector["alpha_ij"]
            xlabel = "Target share alpha_ij"
        else:
            raise ValueError("variable must be 'weight' or 'alpha_ij'.")

        x_sorted, ccdf = ccdf_values(x)

        if len(x_sorted) == 0:
            continue

        plt.plot(x_sorted, ccdf, marker=".", linestyle="none", markersize=2, label=sector)

    plt.xscale("log")
    plt.yscale("log")
    plt.xlabel(xlabel)
    plt.ylabel("CCDF")
    plt.title(f"Log-log CCDF by sector, {year}, variable = {variable}")
    plt.legend()
    plt.tight_layout()

    out_file = OUTPUT_DIR / f"ccdf_by_sector_{variable}_{year}_{TARGET_MODE}_{SHOCK_NAME}.png"
    plt.savefig(out_file, dpi=300, bbox_inches="tight")
    plt.show()

    print("Saved:", out_file)


plot_ccdf_by_sector(year=2023, variable="weight")
plot_ccdf_by_sector(year=2023, variable="alpha_ij")

# Output for Paper

In [ ]:
theta_focus = 0.70

table_baci_summary = (
    summary_df[summary_df["theta"] == theta_focus]
    [[
        "macro_sector",
        "n_sources",
        "n_targets",
        "n_edges",
        "total_weight",
        "frac_sources_slack_weighted_lt_one",
        "min_slack_weighted",
        "median_slack_weighted",
        "gini_slack_weighted"
    ]]
    .copy()
)

table_baci_summary["total_weight_billion"] = table_baci_summary["total_weight"] / 1e6
table_baci_summary["frac_sources_slack_weighted_lt_one_pct"] = (
    100 * table_baci_summary["frac_sources_slack_weighted_lt_one"]
)

table_baci_summary = table_baci_summary[
    [
        "macro_sector",
        "n_sources",
        "n_targets",
        "n_edges",
        "total_weight_billion",
        "min_slack_weighted",
        "median_slack_weighted",
        "frac_sources_slack_weighted_lt_one_pct",
        "gini_slack_weighted"
    ]
].round(4)

display(table_baci_summary)
print(table_baci_summary.to_markdown(index=False))

In [ ]:
tail_paper = tail_fit_df.copy()

tail_paper = tail_paper[
    [
        "macro_sector",
        "variable",
        "n",
        "alpha_hat",
        "xmin",
        "KS_distance",
        "LR_powerlaw_vs_lognormal",
        "p_powerlaw_vs_lognormal",
        "LR_powerlaw_vs_exponential",
        "p_powerlaw_vs_exponential"
    ]
].copy()

tail_paper["variable"] = tail_paper["variable"].replace({
    "alpha_ij": "pi_ij"
})

tail_paper = tail_paper.round(4)

display(tail_paper)
print(tail_paper.to_markdown(index=False))

In [ ]:
theta_values_paper = [0.5, 0.7, 0.9]

theta_sensitivity = (
    summary_df[summary_df["theta"].isin(theta_values_paper)]
    .pivot_table(
        index="macro_sector",
        columns="theta",
        values=[
            "min_slack_weighted",
            "frac_sources_slack_weighted_lt_one"
        ],
        aggfunc="first"
    )
)

# Convert fraction to percentage
for th in theta_values_paper:
    theta_sensitivity[("frac_sources_slack_weighted_lt_one", th)] *= 100

theta_sensitivity = theta_sensitivity.round(4)

display(theta_sensitivity)
print(theta_sensitivity.to_markdown())

In [ ]:
theta_focus = 0.70
year_focus = YEARS[-1]

top3_sources = (
    sources_df[
        (sources_df["theta"] == theta_focus) &
        (sources_df["year"] == year_focus)
    ]
    .sort_values(["macro_sector", "Slack_weighted"], ascending=[True, True])
    .groupby("macro_sector")
    .head(3)
    .reset_index(drop=True)
)

top3_sources = top3_sources[
    [
        "macro_sector",
        "source_name",
        "Slack_weighted",
        "dominance_score",
        "TargetCount",
        "total_weight",
        "n_pivotal_edges",
        "max_alpha"
    ]
].copy()

top3_sources["total_weight_billion"] = top3_sources["total_weight"] / 1e6

top3_sources = top3_sources[
    [
        "macro_sector",
        "source_name",
        "Slack_weighted",
        "dominance_score",
        "TargetCount",
        "total_weight_billion",
        "n_pivotal_edges",
        "max_alpha"
    ]
].round(4)

display(top3_sources)
print(top3_sources.to_markdown(index=False))

In [ ]:
# ============================================================
# BACI EMPIRICAL TARGET CONCENTRATION
# Inputs for final Monte Carlo topology-adjusted calibration
# ============================================================
# This preserves the original target-share, HHI, and max-share
# calculations, without the superseded BACI-degree sigma search.

CALIBRATION_YEAR = 2023

if CALIBRATION_YEAR not in aggregated_edges_by_year:
    raise ValueError(
        f"Year {CALIBRATION_YEAR} not found "
        "in aggregated_edges_by_year."
    )

edges_calibration = (
    aggregated_edges_by_year[
        CALIBRATION_YEAR
    ].copy()
)

# Keep only the six sectors used in the paper
edges_calibration = edges_calibration[
    edges_calibration["macro_sector"].isin(
        SELECTED_SECTORS
    )
].copy()


# Basic cleaning
edges_calibration["weight"] = pd.to_numeric(
    edges_calibration["weight"],
    errors="coerce"
)

edges_calibration = edges_calibration.dropna(
    subset=[
        "macro_sector",
        "target",
        "weight"
    ]
)

edges_calibration = edges_calibration[
    np.isfinite(edges_calibration["weight"])
    & (edges_calibration["weight"] > 0)
].copy()


print(
    "Calibration edges:",
    len(edges_calibration)
)

print(
    "Calibration targets:",
    edges_calibration["target"].nunique()
)


# ============================================================
# 2. EMPIRICAL TARGET SHARES alpha_ij
# ============================================================

# Target totals.
#
# Include macro_sector in grouping for complete safety even
# though each HS6 target should belong to one macro-sector.

edges_calibration["T_j"] = (
    edges_calibration
    .groupby(
        ["macro_sector", "target"]
    )["weight"]
    .transform("sum")
)


# Target share
edges_calibration["alpha_ij"] = (
    edges_calibration["weight"]
    / edges_calibration["T_j"]
)


# ============================================================
# 3. TARGET-LEVEL EMPIRICAL CONCENTRATION
# ============================================================

target_concentration = (
    edges_calibration
    .groupby(
        ["macro_sector", "target"],
        as_index=False
    )
    .agg(
        target_degree=(
            "alpha_ij",
            "size"
        ),

        HHI=(
            "alpha_ij",
            lambda z: np.sum(
                np.square(z)
            )
        ),

        max_share=(
            "alpha_ij",
            "max"
        )
    )
)


# Sanity checks
target_concentration["alpha_equal_share"] = (
    1 /
    target_concentration["target_degree"]
)


print(
    "\nTarget concentration data:"
)

display(
    target_concentration.head()
)


# ============================================================
# 4. EMPIRICAL SUMMARY — POOLED BACI
# ============================================================

# Targets receive equal weight here.
#
# We are characterizing a typical BACI target rather than
# weighting large-degree products more heavily.

empirical_hhi_mean = (
    target_concentration["HHI"].mean()
)

empirical_hhi_median = (
    target_concentration["HHI"].median()
)

empirical_max_mean = (
    target_concentration["max_share"].mean()
)

empirical_max_median = (
    target_concentration["max_share"].median()
)

empirical_degree_mean = (
    target_concentration["target_degree"].mean()
)

empirical_degree_median = (
    target_concentration["target_degree"].median()
)


print(
    "\n==============================================="
)
print(
    "EMPIRICAL BACI TARGET CONCENTRATION"
)
print(
    "==============================================="
)

print(
    f"Number of targets = "
    f"{len(target_concentration)}"
)

print(
    f"Mean target degree = "
    f"{empirical_degree_mean:.3f}"
)

print(
    f"Median target degree = "
    f"{empirical_degree_median:.3f}"
)

print(
    f"Mean HHI = "
    f"{empirical_hhi_mean:.6f}"
)

print(
    f"Median HHI = "
    f"{empirical_hhi_median:.6f}"
)

print(
    f"Mean maximum target share = "
    f"{empirical_max_mean:.6f}"
)

print(
    f"Median maximum target share = "
    f"{empirical_max_median:.6f}"
)


# ============================================================
# 5. EMPIRICAL CONCENTRATION BY SECTOR
# ============================================================

empirical_sector_summary = (
    target_concentration
    .groupby(
        "macro_sector",
        as_index=False
    )
    .agg(
        n_targets=(
            "target",
            "nunique"
        ),

        mean_degree=(
            "target_degree",
            "mean"
        ),

        median_degree=(
            "target_degree",
            "median"
        ),

        mean_HHI=(
            "HHI",
            "mean"
        ),

        median_HHI=(
            "HHI",
            "median"
        ),

        mean_max_share=(
            "max_share",
            "mean"
        ),

        median_max_share=(
            "max_share",
            "median"
        )
    )
)


print(
    "\n--- EMPIRICAL CONCENTRATION BY SECTOR ---"
)

display(
    empirical_sector_summary
)



# Save empirical target-level concentration inputs.
empirical_concentration_file = OUTPUT_DIR / f"empirical_target_concentration_{TARGET_MODE}_{SHOCK_NAME}.csv"
empirical_sector_file = OUTPUT_DIR / f"empirical_target_concentration_by_sector_{TARGET_MODE}_{SHOCK_NAME}.csv"
target_concentration.to_csv(empirical_concentration_file, index=False)
empirical_sector_summary.to_csv(empirical_sector_file, index=False)
print("Saved:", empirical_concentration_file)
print("Saved:", empirical_sector_file)


In [ ]:
# ============================================================
# FINAL LOGNORMAL CALIBRATION:
# BACI CONCENTRATION + ACTUAL MONTE CARLO TOPOLOGY
# ============================================================
#
# PURPOSE
# -------
# Find the Lognormal sigma that reproduces the empirical BACI
# within-target concentration when weights are placed on the
# SAME Bernoulli topology used in the paper's Monte Carlo.
#
#
# EMPIRICAL TARGET TO MATCH:
#
#   BACI mean target HHI
#   BACI mean maximum exporter share
#
#
# SIMULATION TOPOLOGY:
#
#   9 configurations:
#
#   (140,60), (130,70), ..., (60,140)
#
#   p = 20 / M
#
# Thus for every target j:
#
#   degree_j ~ Binomial(M, 20/M)
#
# and E[degree_j] = 20.
#
#
# WHY THIS IS THE FINAL CALIBRATION:
#
# BACI targets have mean degree ~104, while the Monte Carlo
# targets have expected degree 20.
#
# Therefore a dispersion fitted under the BACI degree distribution
# should not be directly transferred into the Monte Carlo.
#
# Here we estimate the sigma that gives BACI-like concentration
# UNDER THE MONTE CARLO TOPOLOGY.
#
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# 0. CHECK THAT PREVIOUS BACI CALIBRATION EXISTS
# ============================================================

if "target_concentration" not in globals():
    raise NameError(
        "target_concentration is not available. "
        "Run the previous BACI empirical target-concentration "
        "chunk first."
    )


# ============================================================
# 1. EMPIRICAL BACI TARGET VALUES TO MATCH
# ============================================================

EMPIRICAL_HHI = float(
    target_concentration["HHI"].mean()
)

EMPIRICAL_MAX_SHARE = float(
    target_concentration["max_share"].mean()
)


print("===============================================")
print("EMPIRICAL BACI CONCENTRATION TARGETS")
print("===============================================")

print(
    f"Mean HHI = "
    f"{EMPIRICAL_HHI:.6f}"
)

print(
    f"Mean maximum share = "
    f"{EMPIRICAL_MAX_SHARE:.6f}"
)


# ============================================================
# 2. MONTE CARLO CONFIGURATIONS FROM THE PAPER
# ============================================================

SCENARIOS = [
    (140, 60),
    (130, 70),
    (120, 80),
    (110, 90),
    (100, 100),
    (90, 110),
    (80, 120),
    (70, 130),
    (60, 140),
]

TARGET_DEGREE = 20


def p_rule(M):
    return TARGET_DEGREE / M


print("\n===============================================")
print("SIMULATION TOPOLOGY")
print("===============================================")

for M, N in SCENARIOS:

    p = p_rule(M)

    print(
        f"(M,N)=({M},{N}) | "
        f"p={p:.6f} | "
        f"E[target degree]={M*p:.2f}"
    )


# ============================================================
# 3. CALIBRATION SETTINGS
# ============================================================
#
# Calibration uses R=300 topology replications.
#
# We use common random numbers:
# the SAME topology degrees and SAME standard-normal draws are
# reused for every candidate sigma.
#
# This makes the objective curve very smooth and prevents
# random Monte Carlo noise from determining the optimum.
#
# After finding sigma, we perform a separate validation with
# R=1000, matching the final paper simulation size.
#
# ============================================================

R_CALIBRATION = 300

R_VALIDATION = 1000

SEED_CALIBRATION = 20261019

SEED_VALIDATION = 20261020


# ============================================================
# 4. GENERATE MONTE CARLO TOPOLOGY + COMMON RANDOM DRAWS
# ============================================================
#
# Important:
#
# The original simulation creates each source-target edge
# independently with probability p.
#
# Consequently, for any target:
#
#       degree_j ~ Binomial(M, p)
#
# Sampling degrees from this Binomial is EXACTLY the marginal
# target-degree distribution generated by the original topology.
#
# For target-level HHI and maximum share, source identities are
# irrelevant once target degree is known.
#
# ============================================================

def generate_calibration_draws(
    scenarios,
    R,
    seed
):

    rng = np.random.default_rng(seed)

    scenario_draws = {}

    topology_summary = []

    for M, N in scenarios:

        p = p_rule(M)

        # ----------------------------------------------------
        # R simulated graphs, each containing N targets
        # ----------------------------------------------------

        degrees = rng.binomial(
            n=M,
            p=p,
            size=R * N
        )

        # Zero-degree targets do not enter the weighted network
        degrees = degrees[
            degrees > 0
        ]

        unique_degrees, counts = np.unique(
            degrees,
            return_counts=True
        )

        # ----------------------------------------------------
        # Generate common standard normal draws Z.
        #
        # Later:
        #
        #   log W = sigma * Z
        #
        # The same Z is reused for every candidate sigma.
        # ----------------------------------------------------

        Z_by_degree = {}

        for d, count in zip(
            unique_degrees,
            counts
        ):

            Z_by_degree[int(d)] = (
                rng.standard_normal(
                    size=(
                        int(count),
                        int(d)
                    )
                )
            )

        scenario_draws[(M, N)] = (
            Z_by_degree
        )

        topology_summary.append(
            {
                "M": M,
                "N": N,
                "p": p,

                "n_simulated_targets":
                    len(degrees),

                "mean_realized_target_degree":
                    degrees.mean(),

                "sd_realized_target_degree":
                    degrees.std(),

                "min_target_degree":
                    degrees.min(),

                "max_target_degree":
                    degrees.max(),
            }
        )

    topology_summary = pd.DataFrame(
        topology_summary
    )

    return (
        scenario_draws,
        topology_summary
    )


scenario_draws, topology_summary = (
    generate_calibration_draws(
        scenarios=SCENARIOS,
        R=R_CALIBRATION,
        seed=SEED_CALIBRATION
    )
)


print(
    "\n--- CALIBRATION TOPOLOGY SUMMARY ---"
)

display(
    topology_summary
)


# ============================================================
# 5. CALCULATE CONCENTRATION FOR ONE SIGMA
# ============================================================

def concentration_for_sigma(
    sigma,
    scenario_draws
):

    scenario_results = []

    for (M, N), Z_by_degree in (
        scenario_draws.items()
    ):

        total_HHI = 0.0
        total_max_share = 0.0
        total_targets = 0


        for d, Z in Z_by_degree.items():

            # ------------------------------------------------
            # Lognormal weights:
            #
            # W = exp(sigma * Z)
            #
            # mu is irrelevant for shares because multiplying
            # every weight by a common constant cancels out.
            #
            # ------------------------------------------------

            log_w = sigma * Z


            # ------------------------------------------------
            # Numerical stabilization
            # ------------------------------------------------

            log_w = (
                log_w
                - np.max(
                    log_w,
                    axis=1,
                    keepdims=True
                )
            )


            weights = np.exp(
                log_w
            )


            shares = (
                weights
                /
                np.sum(
                    weights,
                    axis=1,
                    keepdims=True
                )
            )


            HHI = np.sum(
                shares ** 2,
                axis=1
            )


            max_share = np.max(
                shares,
                axis=1
            )


            total_HHI += np.sum(
                HHI
            )

            total_max_share += np.sum(
                max_share
            )

            total_targets += len(
                HHI
            )


        scenario_results.append(
            {
                "M": M,
                "N": N,

                "sigma": sigma,

                "mean_HHI":
                    total_HHI
                    / total_targets,

                "mean_max_share":
                    total_max_share
                    / total_targets,

                "n_targets":
                    total_targets,
            }
        )


    scenario_df = pd.DataFrame(
        scenario_results
    )


    # --------------------------------------------------------
    # IMPORTANT:
    #
    # Equal-weight the 9 network configurations.
    #
    # Otherwise configurations with larger N would contribute
    # more targets and dominate calibration.
    #
    # --------------------------------------------------------

    mean_HHI_equal_scenarios = (
        scenario_df[
            "mean_HHI"
        ].mean()
    )

    mean_max_equal_scenarios = (
        scenario_df[
            "mean_max_share"
        ].mean()
    )


    return {
        "sigma": sigma,

        "sim_mean_HHI":
            mean_HHI_equal_scenarios,

        "sim_mean_max_share":
            mean_max_equal_scenarios,

        "scenario_results":
            scenario_df,
    }


# ============================================================
# 6. OBJECTIVE FUNCTION
# ============================================================
#
# Relative squared error:
#
# objective =
#
#   [(HHI_sim - HHI_BACI) / HHI_BACI]^2
#
#       +
#
#   [(Max_sim - Max_BACI) / Max_BACI]^2
#
# ============================================================

def calibration_objective(
    sim_HHI,
    sim_max_share
):

    relative_error_HHI = (
        sim_HHI
        - EMPIRICAL_HHI
    ) / EMPIRICAL_HHI


    relative_error_max = (
        sim_max_share
        - EMPIRICAL_MAX_SHARE
    ) / EMPIRICAL_MAX_SHARE


    objective = (
        relative_error_HHI ** 2
        +
        relative_error_max ** 2
    )


    return (
        objective,
        relative_error_HHI,
        relative_error_max
    )


# ============================================================
# 7. COARSE SEARCH
# ============================================================
#
# Broad enough to ensure we do not force the expected result.
#
# ============================================================

SIGMA_GRID_COARSE = np.arange(
    0.50,
    2.51,
    0.10
)


coarse_rows = []


print(
    "\nRunning coarse calibration..."
)


for sigma in SIGMA_GRID_COARSE:

    result = concentration_for_sigma(
        sigma=sigma,
        scenario_draws=scenario_draws
    )


    (
        objective,
        error_HHI,
        error_max
    ) = calibration_objective(
        result["sim_mean_HHI"],
        result["sim_mean_max_share"]
    )


    coarse_rows.append(
        {
            "sigma": sigma,

            "sim_mean_HHI":
                result["sim_mean_HHI"],

            "sim_mean_max_share":
                result["sim_mean_max_share"],

            "relative_error_HHI":
                error_HHI,

            "relative_error_max_share":
                error_max,

            "objective":
                objective,
        }
    )


coarse_df = pd.DataFrame(
    coarse_rows
)


best_coarse_row = (
    coarse_df
    .sort_values(
        "objective"
    )
    .iloc[0]
)


BEST_COARSE_SIGMA = float(
    best_coarse_row[
        "sigma"
    ]
)


print(
    f"Best coarse sigma = "
    f"{BEST_COARSE_SIGMA:.4f}"
)


# ============================================================
# 8. FINE SEARCH
# ============================================================

fine_lower = max(
    0.01,
    BEST_COARSE_SIGMA - 0.15
)

fine_upper = (
    BEST_COARSE_SIGMA + 0.15
)


SIGMA_GRID_FINE = np.arange(
    fine_lower,
    fine_upper + 0.001,
    0.01
)


fine_rows = []


print(
    f"Fine search from "
    f"{fine_lower:.3f} "
    f"to {fine_upper:.3f}"
)


for sigma in SIGMA_GRID_FINE:

    result = concentration_for_sigma(
        sigma=sigma,
        scenario_draws=scenario_draws
    )


    (
        objective,
        error_HHI,
        error_max
    ) = calibration_objective(
        result["sim_mean_HHI"],
        result["sim_mean_max_share"]
    )


    fine_rows.append(
        {
            "sigma": sigma,

            "sim_mean_HHI":
                result["sim_mean_HHI"],

            "sim_mean_max_share":
                result["sim_mean_max_share"],

            "relative_error_HHI":
                error_HHI,

            "relative_error_max_share":
                error_max,

            "objective":
                objective,
        }
    )


fine_df = pd.DataFrame(
    fine_rows
)


best_fine_row = (
    fine_df
    .sort_values(
        "objective"
    )
    .iloc[0]
)


BEST_SIGMA_FINAL = float(
    best_fine_row[
        "sigma"
    ]
)


# ============================================================
# 9. MU FOR SIMULATED MEAN WEIGHT = 100
# ============================================================
#
# If:
#
#   W ~ Lognormal(mu, sigma)
#
# then:
#
#   E[W] = exp(mu + sigma^2 / 2)
#
# Setting E[W] = 100:
#
#   mu = log(100) - sigma^2 / 2
#
# Again, mu does NOT affect normalized shares or Slack.
#
# ============================================================

BEST_MU_FINAL = (
    np.log(100)
    - 0.5
    * BEST_SIGMA_FINAL ** 2
)


# ============================================================
# 10. CALIBRATION RESULTS AT BEST SIGMA
# ============================================================

best_calibration_result = (
    concentration_for_sigma(
        sigma=BEST_SIGMA_FINAL,
        scenario_draws=scenario_draws
    )
)


best_scenario_df = (
    best_calibration_result[
        "scenario_results"
    ].copy()
)


best_scenario_df[
    "empirical_BACI_mean_HHI"
] = EMPIRICAL_HHI


best_scenario_df[
    "empirical_BACI_mean_max_share"
] = EMPIRICAL_MAX_SHARE


# ============================================================
# 11. INDEPENDENT VALIDATION WITH R = 1000
# ============================================================
#
# The calibration used R=300.
#
# Now generate NEW random topology/weights with a new seed and
# R=1000, matching the final Monte Carlo design.
#
# This verifies that the selected sigma is not an artifact of
# the particular calibration random draws.
#
# ============================================================

def validate_final_sigma(
    sigma,
    scenarios,
    R,
    seed
):

    rng = np.random.default_rng(
        seed
    )

    validation_rows = []


    for M, N in scenarios:

        p = p_rule(M)


        degrees = rng.binomial(
            n=M,
            p=p,
            size=R * N
        )


        degrees = degrees[
            degrees > 0
        ]


        unique_degrees, counts = (
            np.unique(
                degrees,
                return_counts=True
            )
        )


        total_HHI = 0.0
        total_max = 0.0
        total_targets = 0


        for d, count in zip(
            unique_degrees,
            counts
        ):

            Z = rng.standard_normal(
                size=(
                    int(count),
                    int(d)
                )
            )


            log_w = sigma * Z


            log_w = (
                log_w
                - np.max(
                    log_w,
                    axis=1,
                    keepdims=True
                )
            )


            w = np.exp(
                log_w
            )


            shares = (
                w
                /
                np.sum(
                    w,
                    axis=1,
                    keepdims=True
                )
            )


            HHI = np.sum(
                shares ** 2,
                axis=1
            )


            max_share = np.max(
                shares,
                axis=1
            )


            total_HHI += np.sum(
                HHI
            )


            total_max += np.sum(
                max_share
            )


            total_targets += (
                len(HHI)
            )


        validation_rows.append(
            {
                "M": M,
                "N": N,

                "p": p,

                "mean_target_degree":
                    degrees.mean(),

                "mean_HHI":
                    total_HHI
                    / total_targets,

                "mean_max_share":
                    total_max
                    / total_targets,

                "n_targets":
                    total_targets,
            }
        )


    validation_df = pd.DataFrame(
        validation_rows
    )


    # Equal-weight the nine scenarios
    overall_HHI = (
        validation_df[
            "mean_HHI"
        ].mean()
    )


    overall_max = (
        validation_df[
            "mean_max_share"
        ].mean()
    )


    return (
        validation_df,
        overall_HHI,
        overall_max
    )


(
    validation_df,
    VALIDATION_HHI,
    VALIDATION_MAX_SHARE
) = validate_final_sigma(
    sigma=BEST_SIGMA_FINAL,
    scenarios=SCENARIOS,
    R=R_VALIDATION,
    seed=SEED_VALIDATION
)


# ============================================================
# 12. FINAL SUMMARY
# ============================================================

(
    validation_objective,
    validation_error_HHI,
    validation_error_max
) = calibration_objective(
    VALIDATION_HHI,
    VALIDATION_MAX_SHARE
)


print(
    "\n==============================================="
)

print(
    "FINAL MONTE CARLO-TOPOLOGY "
    "BACI LOGNORMAL CALIBRATION"
)

print(
    "==============================================="
)


print(
    f"Best sigma = "
    f"{BEST_SIGMA_FINAL:.6f}"
)


print(
    f"mu for arithmetic mean weight 100 = "
    f"{BEST_MU_FINAL:.6f}"
)


print(
    "\n--- BACI TARGET ---"
)

print(
    f"Mean HHI = "
    f"{EMPIRICAL_HHI:.6f}"
)

print(
    f"Mean max share = "
    f"{EMPIRICAL_MAX_SHARE:.6f}"
)


print(
    "\n--- CALIBRATION SAMPLE (R=300) ---"
)

print(
    f"Mean HHI = "
    f"{best_calibration_result['sim_mean_HHI']:.6f}"
)

print(
    f"Mean max share = "
    f"{best_calibration_result['sim_mean_max_share']:.6f}"
)


print(
    "\n--- INDEPENDENT VALIDATION (R=1000) ---"
)

print(
    f"Mean HHI = "
    f"{VALIDATION_HHI:.6f}"
)

print(
    f"Mean max share = "
    f"{VALIDATION_MAX_SHARE:.6f}"
)


print(
    f"\nValidation relative HHI error = "
    f"{100 * validation_error_HHI:.3f}%"
)

print(
    f"Validation relative max-share error = "
    f"{100 * validation_error_max:.3f}%"
)

print(
    f"Validation objective = "
    f"{validation_objective:.8f}"
)


print(
    "\n--- VALIDATION BY SIMULATION CONFIGURATION ---"
)

display(
    validation_df
)


# ============================================================
# 13. FINAL ONE-ROW TABLE
# ============================================================

final_calibration_summary = (
    pd.DataFrame(
        [
            {
                "BACI_mean_HHI":
                    EMPIRICAL_HHI,

                "BACI_mean_max_share":
                    EMPIRICAL_MAX_SHARE,

                "best_sigma":
                    BEST_SIGMA_FINAL,

                "mu_for_mean_100":
                    BEST_MU_FINAL,

                "calibration_R":
                    R_CALIBRATION,

                "calibration_mean_HHI":
                    best_calibration_result[
                        "sim_mean_HHI"
                    ],

                "calibration_mean_max_share":
                    best_calibration_result[
                        "sim_mean_max_share"
                    ],

                "validation_R":
                    R_VALIDATION,

                "validation_mean_HHI":
                    VALIDATION_HHI,

                "validation_mean_max_share":
                    VALIDATION_MAX_SHARE,

                "validation_rel_error_HHI":
                    validation_error_HHI,

                "validation_rel_error_max_share":
                    validation_error_max,

                "validation_objective":
                    validation_objective,
            }
        ]
    )
)


display(
    final_calibration_summary
)


# ============================================================
# 14. SAVE EVERYTHING
# ============================================================

final_grid_df = pd.concat(
    [
        coarse_df.assign(
            search="coarse"
        ),

        fine_df.assign(
            search="fine"
        )
    ],
    ignore_index=True
)


grid_file = (
    OUTPUT_DIR
    /
    f"lognormal_FINAL_topology_calibration_grid_"
    f"{TARGET_MODE}_{SHOCK_NAME}.csv"
)


summary_file = (
    OUTPUT_DIR
    /
    f"lognormal_FINAL_topology_calibration_"
    f"{TARGET_MODE}_{SHOCK_NAME}.csv"
)


scenario_file = (
    OUTPUT_DIR
    /
    f"lognormal_FINAL_topology_validation_by_scenario_"
    f"{TARGET_MODE}_{SHOCK_NAME}.csv"
)


topology_file = (
    OUTPUT_DIR
    /
    f"lognormal_FINAL_topology_summary_"
    f"{TARGET_MODE}_{SHOCK_NAME}.csv"
)


final_grid_df.to_csv(
    grid_file,
    index=False
)


final_calibration_summary.to_csv(
    summary_file,
    index=False
)


validation_df.to_csv(
    scenario_file,
    index=False
)


topology_summary.to_csv(
    topology_file,
    index=False
)


print(
    "\nSaved:"
)

print(grid_file)
print(summary_file)
print(scenario_file)
print(topology_file)


# ============================================================
# 15. OBJECTIVE PLOT
# ============================================================

plot_df = (
    fine_df
    .sort_values(
        "sigma"
    )
)


fig, ax = plt.subplots(
    figsize=(8, 5)
)


ax.plot(
    plot_df["sigma"],
    plot_df["objective"],
    marker="o",
    markersize=3
)


ax.axvline(
    BEST_SIGMA_FINAL,
    linestyle="--"
)


ax.set_xlabel(
    "Lognormal sigma"
)


ax.set_ylabel(
    "Calibration objective"
)


ax.set_title(
    "Final BACI Lognormal calibration\n"
    "under Monte Carlo topology"
)


ax.grid(
    alpha=0.25
)


plt.tight_layout()


plot_file = (
    OUTPUT_DIR
    /
    f"lognormal_FINAL_topology_calibration_"
    f"{TARGET_MODE}_{SHOCK_NAME}.png"
)


plt.savefig(
    plot_file,
    dpi=300,
    bbox_inches="tight"
)


plt.show()


print(
    "Saved plot:",
    plot_file
)


# ============================================================
# 16. FINAL PARAMETERS FOR simulation_REV1.py
# ============================================================

print(
    "\n================================================"
)

print(
    "FINAL PARAMETERS FOR simulation_REV1.py"
)

print(
    "================================================"
)


print(
    f"LOGNORMAL_SIGMA_BACI = "
    f"{BEST_SIGMA_FINAL:.8f}"
)


print(
    f"LOGNORMAL_MU_BACI = "
    f"{BEST_MU_FINAL:.8f}"
)